# LightGBM Regressor — Housing Rent Prediction

Standalone model notebook. It runs the **identical** data preparation as every other model (via `src/`), trains **LightGBM Regressor**, evaluates on validation with 5-fold CV on the training set, and writes results to `reports/` for comparison.

- **Target:** `Rent`, modelled on `log1p` scale; predictions reported in rupees.
- **Preprocessing:** no scaling (tree-based / scale-invariant), inside a Pipeline fit on the training set only.
- **Test set:** untouched while `EVALUATE_TEST = False` (flip to `True` only for final evaluation).
- **Split:** fixed 70/15/15, `random_state=42`, shared across all models.

In [ ]:
# LightGBM needs the package (and libomp on macOS: `brew install libomp`).
!pip install lightgbm

In [ ]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # put repo root on the path so `import src` works
warnings.filterwarnings('ignore')
from src.preprocessing import prepare_data, assert_run_consistency
from src.models import make_model, display_name
from src.evaluation import evaluate, results_table, print_summary, save_results, plot_diagnostics
from src.prediction import save_model
from src import config

In [ ]:
MODEL_KEY = 'lightgbm_model'
EVALUATE_TEST = False   # keep False until the final-evaluation stage

In [ ]:
# Identical data preparation for EVERY model: reads House_Rent_Dataset.csv and runs the frozen
# src pipeline (clean -> features -> 70/15/15 split -> fit-on-train encode). Deterministic by seed.
data = prepare_data()                       # data/raw/House_Rent_Dataset.csv
ref = assert_run_consistency(data)          # fairness tripwire: all models must share this reference_id
n_rows = sum(data['meta']['rows'][s] for s in ('train', 'val', 'test'))
print(f"Prepared: {data['X_train'].shape[0]} train / {data['X_val'].shape[0]} val / "
      f"{data['X_test'].shape[0]} test | {len(data['feature_names'])} features | reference_id {ref}")

### The only model-specific line

In [ ]:
model = make_model(MODEL_KEY, data['target_transform'])
model

### Train, validate, cross-validate

In [ ]:
results, pred, train_time = evaluate(model, data, display_name(MODEL_KEY), MODEL_KEY,
                                     evaluate_test=EVALUATE_TEST)
print_summary(results, n_rows, len(data['feature_names']))
results_table(results)

In [ ]:
save_results(results, pred, data)           # -> reports/metrics/ and reports/predictions/
save_model(model, MODEL_KEY)                # -> models/<key>.joblib
print('Saved: reports/metrics, reports/predictions, models/')

### Diagnostics

In [ ]:
plot_diagnostics(results, pred, data, model, display_name(MODEL_KEY));